# 01 · Clean the student dataset
Run the numbered notebooks in order, and run cells from top to bottom using **Run All**. Choose the project Python environment; install `requirements.txt` and `ipykernel` if needed.

We convert month names and category labels to numeric codes, keep one row per student, and save **cleaned_dataset.xlsx**. The original spreadsheet is preserved. Tables below show previews; saved files contain all rows.


## 1. Load the original spreadsheet


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Open the notebook from the project folder. All paths are relative to it.
BASE_DIR = Path.cwd()
if not (BASE_DIR / "datathon-dataset-2026.xlsx").exists():
    raise FileNotFoundError("Open this notebook with the datathon project folder as the working directory.")
print(f"Project folder: {BASE_DIR}")

dataset = pd.read_excel(BASE_DIR / "datathon-dataset-2026.xlsx")
original_rows = len(dataset)
display(pd.DataFrame({"rows": [len(dataset)], "columns": [dataset.shape[1]]}))
display(dataset.head())


Project folder: c:\Users\Grcar\Documents\datathon


,rows,columns
0,69557,67


,student_id,entry_year,credit_hrs,app_dt,app_mo,app_yr,app_fee,acceptance_fee,hs_gpa,act_cmpst,...,hs_only_math,hs_only_sci,rotc_interest,miles_from_campus,veteran,age,stem_major,urm,sex,college
0,145382.0,2022,0.0,13.0,nov,2021,Deferred,NaN,2.15,11.0,...,8.0,8.0,0,855.0,0,20.0,0,1,M,Public Health and Translational Science
1,89694.0,2022,0.0,27.0,dec,2021,Deferred,NaN,2.14,11.0,...,NaN,4.0,0,307.0,0,18.0,0,1,M,Public Health and Translational Science
2,112761.0,2022,0.0,27.0,nov,2021,Deferred,NaN,2.67,11.0,...,NaN,NaN,0,534.0,0,19.0,1,1,F,Public Health and Translational Science
3,61545.0,2022,0.0,11.0,oct,2021,Deferred,NaN,2.84,11.0,...,8.0,7.0,0,29.0,0,18.0,0,0,M,Liberal Arts and Sciences
4,113888.0,2022,0.0,9.0,aug,2021,Paid,NaN,2.22,12.0,...,NaN,NaN,0,226.0,0,18.0,0,1,M,"Business, Finance and Industry"


## 2. Convert month names to numbers
January becomes 1, February becomes 2, and so on. Unrecognized or missing names become missing values, matching the original cleaner.


In [2]:
months = {
    "jan": 1,
    "feb": 2,
    "mar": 3,
    "apr": 4,
    "may": 5,
    "jun": 6,
    "jul": 7,
    "aug": 8,
    "sep": 9,
    "oct": 10,
    "nov": 11,
    "dec": 12
}

dataset["app_mo"] = dataset["app_mo"].str.lower().map(months)
dataset["first_contact_mo"] = dataset["first_contact_mo"].str.lower().map(months)
dataset["visit_mo"] = dataset["visit_mo"].str.lower().map(months)

display(dataset[["app_mo", "first_contact_mo", "visit_mo"]].head(10))


,app_mo,first_contact_mo,visit_mo
0,11,11.0,NaN
1,12,12.0,NaN
2,11,11.0,NaN
3,10,5.0,NaN
4,8,8.0,NaN
5,11,11.0,NaN
6,2,1.0,NaN
7,2,1.0,NaN
8,2,11.0,NaN
9,2,3.0,NaN


## 3. Encode categories and show the lookup
Codes follow first appearance in this spreadsheet. They are labels, not an ordering; `-1` means missing. Keep the lookup when interpreting the charts or cleaning future data.


In [3]:
category_lookup_rows = []
for column in ["app_fee", "acceptance_fee", "residency", "sex", "college"]:
    codes, labels = pd.factorize(dataset[column])
    category_lookup_rows.extend({"column": column, "code": code, "original_label": str(label)} for code, label in enumerate(labels))
    category_lookup_rows.append({"column": column, "code": -1, "original_label": "Missing / not recorded"})
    dataset[column] = codes
category_lookup = pd.DataFrame(category_lookup_rows)
display(category_lookup)


,column,code,original_label
0,app_fee,0,Deferred
1,app_fee,1,Paid
2,app_fee,2,Waived
3,app_fee,-1,Missing / not recorded
4,acceptance_fee,0,Paid
5,acceptance_fee,1,Deferred
6,acceptance_fee,2,Waived
7,acceptance_fee,-1,Missing / not recorded
8,residency,0,Non-Contiguous
9,residency,1,Contiguous


## 4. Keep one row per student
Drop rows without a student ID, then keep the first occurrence of each ID.


In [4]:
missing_id_rows = int(dataset["student_id"].isna().sum())
with_ids = dataset.dropna(subset=["student_id"])
duplicate_rows = int(with_ids.duplicated(subset="student_id").sum())
dataset = with_ids.drop_duplicates(subset="student_id", keep="first").copy()
display(pd.DataFrame({"step": ["Original rows", "Missing ID rows removed", "Duplicate ID rows removed", "Remaining rows"], "rows": [original_rows, missing_id_rows, duplicate_rows, len(dataset)]}))


,step,rows
0,Original rows,69557
1,Missing ID rows removed,2
2,Duplicate ID rows removed,14
3,Remaining rows,69541


## 5. Set numeric types and inspect missingness
Nullable integers retain missing values. GPA and credit hours retain their original numeric precision. The missingness table also counts the `-1` category sentinel.


In [5]:
cols_to_convert = [col for col in dataset.columns if col not in ["hs_gpa", "credit_hrs"]]
dataset[cols_to_convert] = dataset[cols_to_convert].astype("Int64")
missing_counts = dataset.isna().sum()
for column in category_lookup["column"].unique():
    missing_counts[column] += int(dataset[column].eq(-1).sum())
display(pd.DataFrame({"dtype": dataset.dtypes.astype(str), "missing_rows": missing_counts, "missing_percent": missing_counts / len(dataset) * 100}))
display(dataset.head())


,dtype,missing_rows,missing_percent
student_id,Int64,0,0.000000
entry_year,Int64,0,0.000000
credit_hrs,float64,26,0.037388
app_dt,Int64,2313,3.326095
app_mo,Int64,0,0.000000
...,...,...,...
age,Int64,2,0.002876
stem_major,Int64,0,0.000000
urm,Int64,0,0.000000
sex,Int64,0,0.000000


,student_id,entry_year,credit_hrs,app_dt,app_mo,app_yr,app_fee,acceptance_fee,hs_gpa,act_cmpst,...,hs_only_math,hs_only_sci,rotc_interest,miles_from_campus,veteran,age,stem_major,urm,sex,college
0,145382,2022,0.0,13,11,2021,0,-1,2.15,11,...,8,8,0,855,0,20,0,1,0,0
1,89694,2022,0.0,27,12,2021,0,-1,2.14,11,...,<NA>,4,0,307,0,18,0,1,0,0
2,112761,2022,0.0,27,11,2021,0,-1,2.67,11,...,<NA>,<NA>,0,534,0,19,1,1,1,0
3,61545,2022,0.0,11,10,2021,0,-1,2.84,11,...,8,7,0,29,0,18,0,0,0,1
4,113888,2022,0.0,9,8,2021,1,-1,2.22,12,...,<NA>,<NA>,0,226,0,18,0,1,0,2


## 6. Save the cleaned data and category lookup
The next notebook reads this Excel file explicitly, so an older CSV cannot silently replace this run.


In [6]:
cleaned_file = BASE_DIR / "cleaned_dataset.xlsx"
dataset.to_excel(cleaned_file, index=False)
lookup_file = BASE_DIR / "cleaned_category_lookup.csv"
category_lookup.to_csv(lookup_file, index=False)
display(pd.DataFrame({"saved_file": [cleaned_file.name, lookup_file.name], "rows": [len(dataset), len(category_lookup)]}))


,saved_file,rows
0,cleaned_dataset.xlsx,69541
1,cleaned_category_lookup.csv,22


Next: 02_preliminary_analysis.ipynb
